In [116]:
import pandas as pd
import requests
from PIL import Image, ImageDraw, ImageFont
import io
from IPython.display import display, clear_output, Audio
import ipywidgets as widgets
from google.oauth2 import service_account
from google.auth.transport.requests import Request
import base64
import json

In [149]:
# Load credentials from your service account file
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

In [150]:
universes_df = pd.read_csv('data/alternate_universes.csv')

In [151]:
creds.refresh(Request())
token = creds.token

In [152]:
with open('keys/keys.json', 'r') as f:
    keys = json.load(f)

In [153]:
def get_audio(universe):

    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
      "model": "lyria-3-clip-preview",
      "input": [
        {
          "type": "text",
          "text": f"{universe} style game chase battle theme song with tempo speeding up, no lyrics"
        }
      ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload)
    # print(response)
    # print(response.text)
    # print(response.status_code)
    # print(response.headers)
    data = response.json()
    bytedata = data["outputs"][2]["data"]
    return base64.b64decode(bytedata)

In [154]:
def play_announcement(text):
    try:
        mp3 = get_audio(text)
        display(Audio(mp3, autoplay=True))
    except Exception as e:
        print(f"(Couldn't play audio: {e})")

In [156]:
# Create locker buttons
buttons = [
    widgets.Button(
        description=f'#{i}', 
        button_style='info',
        layout=widgets.Layout(
            width="16.6%",
            height="45px"
        )
    )
    for i in range(101, 107)
]

# Load the initial locker image
locker_image = Image.open('assets/Lockers.png').convert("RGBA")

# Draw the placeholder text over the locker image
draw = ImageDraw.Draw(locker_image)
font = ImageFont.load_default(size=40)

message = "Choose a locker to start your adventure..."
bbox = draw.textbbox((0, 0), message, font=font)
text_width = bbox[2] - bbox[0]

text_x = (locker_image.width - text_width) // 2
text_y = 20

draw.text(
    (text_x, text_y),
    message,
    fill="white",
    font=font,
    stroke_width=2,
    stroke_fill="black"
)

# Convert the edited image into a widget
buffer = io.BytesIO()
locker_image.save(buffer, format="PNG")

image_widget = widgets.Image(
    value=buffer.getvalue(),
    format='png'
)

# One output area for the entire interface
output = widgets.Output()

def show_loading_message():
    loading_image = Image.open('assets/Lockers.png').convert("RGBA")
    draw = ImageDraw.Draw(loading_image)

    font = ImageFont.load_default(size=40)
    message = "Transporting through portal, please wait..."

    bbox = draw.textbbox((0, 0), message, font=font)
    text_width = bbox[2] - bbox[0]

    text_x = (loading_image.width - text_width) // 2
    text_y = 20

    draw.text(
        (text_x, text_y),
        message,
        fill="yellow",
        font=font,
        stroke_width=2,
        stroke_fill="black"
    )

    buffer = io.BytesIO()
    loading_image.save(buffer, format="PNG")

    image_widget.value = buffer.getvalue()

def on_battle_click(b):
    show_loading_message()
    my_universe = universes_df.sample(1).iloc[0]
    my_name = my_universe["Universe_Name"]
    my_difficulty = my_universe["Difficulty_Level"]
    my_image = 'assets/' + my_universe["Background_Img"]
    

    # Load images
    background = Image.open(my_image).convert("RGBA")
    pip_original = Image.open('assets/Pip.png').convert("RGBA")
    villain_original = Image.open('assets/Villain.png').convert("RGBA")

    # Resize Pip
    pip_width = int(background.height * 0.3)
    pip_height = int(pip_original.height * pip_width / pip_original.width)
    pip_image = pip_original.resize((pip_width, pip_height))

    # Resize villain
    villain_width = int(background.height * 0.3)
    villain_height = int(
        villain_original.height * villain_width / villain_original.width
    )
    villain_image = villain_original.resize((villain_width, villain_height))

    # Starting positions
    pip_x = 20
    pip_y = background.height - pip_height - 130
    villain_x = background.width - villain_width - 20
    villain_y = background.height - villain_height - 130

    with output:
        clear_output(wait=True)
        play_announcement(my_name)
        # Create a separate display area for the image
        image_output = widgets.Output()

        def show_scene():
            with image_output:
                clear_output(wait=True)
        
                scene = background.copy()
                scene.alpha_composite(pip_image, (pip_x, pip_y))
                scene.alpha_composite(villain_image, (villain_x, villain_y))
        
                draw = ImageDraw.Draw(scene)
                font = ImageFont.load_default(size=40)
        
                # Display the universe name
                main_message = f"Welcome to {my_name}"
                bbox = draw.textbbox((0, 0), main_message, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
        
                text_x1 = (scene.width - text_width) // 2
                text_y1 = 20
        
                draw.text(
                    (text_x1, 20),
                    main_message,
                    fill="white",
                    font=font,
                    stroke_width=2,
                    stroke_fill="black"
                )

                # Display the difficulty level
                font2 = ImageFont.load_default(size=20)
                
                difficulty_message = f"Difficulty Level: {my_difficulty}"
                bbox2 = draw.textbbox((0, 0), difficulty_message, font=font2)
                
                text_width2 = bbox2[2] - bbox2[0]
                
                text_x2 = (scene.width - text_width2) // 2
                text_y2 = 65
                
                draw.text(
                    (text_x2, text_y2),
                    difficulty_message,
                    fill="white",
                    font=font2,
                    stroke_width=1,
                    stroke_fill="black"
                )

                # Display the difficulty level
                font4 = ImageFont.load_default(size=20)
                
                difficulty_message = "Keep tapping the button to chase up to the villain"
                bbox4 = draw.textbbox((0, 0), difficulty_message, font=font4)
                
                text_width4 = bbox4[2] - bbox4[0]
                
                text_x4 = (scene.width - text_width4) // 2
                text_y4 = scene.width - 20
                
                draw.text(
                    (text_x4, text_y4),
                    difficulty_message,
                    fill="white",
                    font=font4,
                    stroke_width=1,
                    stroke_fill="black"
                )

                # Display the win message when Pip reaches the villain
                if pip_x + pip_width/2 >= villain_x:
                    message = "You Win!"
                    font3 = ImageFont.load_default(size=60)
                    bbox = draw.textbbox((0, 0), message, font=font3)
                    text_width = bbox[2] - bbox[0]
                    text_height = bbox[3] - bbox[1]
        
                    text_x = (scene.width - text_width) // 2
                    text_y = (scene.height - text_height) // 2
        
                    draw.text(
                        (text_x, text_y),
                        message,
                        fill="yellow",
                        font=font3,
                        stroke_width=3,
                        stroke_fill="black"
                    )
        
                display(scene)
                
        
        def on_chase_click(button):
            nonlocal pip_x
            difficulty_mapped = int(80 - (my_difficulty - 1) * 78 / 98)
        
            # Stop moving once Pip reaches the villain
            if pip_x + pip_width/2 < villain_x:
                pip_x += difficulty_mapped
        
            show_scene()

        button_chase = widgets.Button(
            description="CLICK ME",
            button_style="danger",
            layout=widgets.Layout(
                width="100%",
                height="45px"
            )
        )
        button_chase.on_click(on_chase_click)

        display(widgets.VBox([
            image_output,
            button_chase
        ], layout=widgets.Layout(width="100%")))
        show_scene()

# Connect all buttons
for button in buttons:
    button.on_click(on_battle_click)

# Put everything in the same output area
with output:
    display(widgets.VBox([
        image_widget,
        widgets.HBox(buttons)
    ], layout=widgets.Layout(width="100%")))

display(output)

Output()